# Comprehensive Model Evaluation — iteris

Cross-cutting evaluation of every model this project has produced: U-Net baselines
(**LiteUNet**, **AttentionResUNet**), the **BRISC tumor-type classifier**, and DRL
refiners (**DuelingDDQN** — discrete contour-sector actions, **TD3** — continuous
contour-sector actions) across both datasets (**CAMUS**: LV_endo / LV_epi / LA,
**BRISC**: glioma / meningioma / pituitary / tumor).

This notebook does **not** hardcode any results, and does **not** import `iteris`/
`torch`/`monai` for inference — it only reads whatever run-output *files* you drop
into `results/` (locally) or attach as Kaggle input datasets, and computes every
table/plot/verdict directly from them. If a section has no matching files it prints
what it's missing instead of fabricating numbers. (Optional: if `torch` happens to
be importable, Section 9 additionally reads lightweight scalar metadata out of
`.pt` checkpoints — never rebuilds a model, so `monai`/`iteris` are never required.)

## Where it looks for files

- **Locally**: `results/` (or `../results` / `../../results`, whichever exists),
  searched recursively — subfolders are fine.
- **On Kaggle**: every dataset attached under `/kaggle/input/` is searched
  recursively too, so you can attach all your output datasets (baselines, every
  DRL run, the classifier) to one notebook and just run it — no need to copy
  anything into one folder first. Outputs (figures, exported tables) are written
  to `/kaggle/working/evaluation_outputs/` there, since `/kaggle/input` is read-only.

Every file is identified by its **content shape** (or, for CSVs/PNGs/checkpoints
that carry no metadata of their own, by matching them to a same-folder sibling
JSON with the same filename stem) — never by which folder you put it in, so
filename collisions across separate runs (e.g. BRISC's per-tumor-type DRL configs
all use `target_class=1`) are harmless as long as each run's files live together.

## What's recognized automatically

| File | Produced by | Identified by |
|---|---|---|
| `<dataset><suffix>_summary.json` | `iteris.evaluation.save_summary_json` | `test_dice` key |
| `<dataset><suffix>_test_scores.csv` | `iteris.evaluation.evaluate_test_set` | per-patient `dice_<class>` columns |
| `<dataset><suffix>_learning_curves.png` | `iteris.visualization.plot_learning_curves` | filename |
| `<dataset><suffix>_qualitative.png` | `iteris.visualization.plot_qualitative_grid` | filename |
| `<dataset>_<agent>_c<class>_reeval_test_results.json` | `iteris.drl_reeval.reeval_checkpoint` | `init_dice_mean`+`final_dice_mean` keys |
| `..._reeval_comparison.png` / `..._reeval_behaviour.png` | same (`plot_comparison`/`plot_behaviour`) | filename |
| `brisc_tumor_classifier_summary.json` | `iteris.classifier.save_classifier_summary_json` | `accuracy`+`per_class` keys |
| `brisc_tumor_classifier_learning_curves.png` / `..._confusion_matrix.png` | classifier training/eval notebook | filename |
| `*.pt` checkpoints | any training run | optional, `torch`-only, scalar fields only |

Optional, for real significance testing beyond the U-Net per-patient CSVs above: a
per-sample CSV per **DRL** run with `init_dice`/`final_dice` columns (nothing in
the current pipeline exports this yet — see Section 10).

## Sections
1. Ingestion
2. Master tidy comparison table
3. U-Net baseline landscape (Lite vs Attention — the DRL headroom ceiling)
4. Discrete vs continuous action space (DuelingDDQN vs TD3)
5. Per-class / per-dataset learning behaviour
6. Most-improved DRL agent ranking
7. Qualitative visual gallery (comparison / behaviour / learning-curve / prediction images)
8. BRISC tumor-type classifier evaluation
9. Checkpoint / training provenance (optional, needs `torch`)
10. Statistical significance (Wilcoxon signed-rank + Bonferroni)
11. Export & auto-generated summary


In [ ]:
import json, os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from scipy import stats

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

try:
    import torch
    HAVE_TORCH = True
except ImportError:
    HAVE_TORCH = False


In [ ]:
# Configuration -- where to search for files, where to write outputs.
_local_candidates = [Path('../../results'), Path('results'), Path('../results')]
LOCAL_RESULTS = next((p for p in _local_candidates if p.exists()), None)
KAGGLE_INPUT = Path('/kaggle/input')

SEARCH_ROOTS = []
if LOCAL_RESULTS is not None:
    SEARCH_ROOTS.append(LOCAL_RESULTS)
if KAGGLE_INPUT.exists():
    SEARCH_ROOTS.append(KAGGLE_INPUT)
if not SEARCH_ROOTS:
    SEARCH_ROOTS = [_local_candidates[1]]  # 'results' -- may not exist yet, that's fine

if Path('/kaggle/working').exists():
    OUT_DIR = Path('/kaggle/working/evaluation_outputs')
elif LOCAL_RESULTS is not None:
    OUT_DIR = LOCAL_RESULTS
else:
    OUT_DIR = Path('results')
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('[config] searching for result files under:')
for r in SEARCH_ROOTS:
    print(f'    - {r.resolve()}')
print(f'[config] writing figures/exports to: {OUT_DIR.resolve()}')
print(f'[config] torch available: {HAVE_TORCH}')

def find_files(exts):
    # Exclude only this notebook's own exported figures/table (NOT all of OUT_DIR --
    # locally OUT_DIR often *is* a search root itself, so excluding the whole tree
    # would hide the real input files too).
    fig_dir_resolved = FIG_DIR.resolve()
    out = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for p in root.rglob('*'):
            if not p.is_file() or p.suffix.lower() not in exts:
                continue
            resolved = p.resolve()
            if fig_dir_resolved in resolved.parents:
                continue
            if resolved.name == 'master_comparison.csv':
                continue
            out.append(p)
    return sorted(set(out))

# agent_type (as stored in cfg / saved JSON) -> action space + display name
AGENT_META = {
    'TD3':      dict(action_space='continuous', display='TD3'),
    'DUELING':  dict(action_space='discrete',   display='DuelingDDQN'),
    'DDPG':     dict(action_space='continuous', display='DDPG (archived global)'),
    'DQN':      dict(action_space='discrete',   display='DQN (archived global)'),
}
# internal cfg['model'] key -> display name (matches iteris.evaluation._model_names)
UNET_META = {
    'lite_unet':        'LiteUNet',
    'attn_resunet':     'AttentionResUNet',
    'LiteUNet':         'LiteUNet',
    'AttentionResUNet': 'AttentionResUNet',
}
DATASET_CLASS_ORDER = {
    'CAMUS': ['LV_endo', 'LV_epi', 'LA'],
    'BRISC': ['glioma', 'meningioma', 'pituitary', 'tumor'],
}

# Filename-stem matching so CSVs/PNGs/checkpoints (which carry no metadata of
# their own) can inherit dataset/model/class/agent from a same-folder sibling
# JSON, purely from filename structure -- never by directory naming.
JSON_SUFFIXES = ['_summary.json', '_reeval_test_results.json']

def strip_suffix(name, suffixes):
    for s in suffixes:
        if name.endswith(s):
            return name[:-len(s)]
    return None

def sibling_meta(path, own_suffixes, json_meta):
    """Find a same-folder JSON whose stem matches this file's stem, return its meta dict."""
    stem = strip_suffix(path.name, own_suffixes)
    if stem is None:
        return None
    for jpath_str, meta in json_meta.items():
        jpath = Path(jpath_str)
        if jpath.parent != path.parent:
            continue
        if strip_suffix(jpath.name, JSON_SUFFIXES) == stem:
            return meta
    return None


## 1. Ingestion\n\nWalk every file under the search roots, sniff its shape/name, and route it. Anything unrecognized is reported, not silently dropped.

In [ ]:
def _load_json(path):
    try:
        with open(path, 'r') as f:
            return json.load(f)
    except Exception as e:
        print(f'  [skip] {path}: could not parse ({e})')
        return None

def parse_unet_summary(d, path):
    rows = []
    dataset = d.get('dataset', '?')
    model = UNET_META.get(d.get('model', '?'), d.get('model', '?'))
    for cls, dice in d.get('test_dice', {}).items():
        rows.append(dict(
            source=str(path), dataset=dataset, model=model, model_family='UNet',
            action_space='n/a', class_name=cls, dice=dice,
            iou=d.get('test_iou', {}).get(cls), biou=d.get('test_biou', {}).get(cls),
            precision=d.get('test_precision', {}).get(cls), sensitivity=d.get('test_sensitivity', {}).get(cls),
            hd95=d.get('test_hd95', {}).get(cls), hd95geo=d.get('test_hd95geo_px', {}).get(cls),
            msd=d.get('test_msd_px', {}).get(cls), best_val_dice=d.get('best_val_dice'),
            label_frac=d.get('label_frac'),
        ))
    return rows, dict(dataset=dataset, model=model, kind='unet')

def parse_drl_summary(d, path):
    agent_type = str(d.get('agent_type', '?')).upper()
    meta = AGENT_META.get(agent_type, dict(action_space='unknown', display=agent_type))
    row = dict(
        source=str(path), dataset=d.get('dataset', '?'), class_name=d.get('class_name', '?'),
        target_class=d.get('target_class'), model=meta['display'], model_family='DRL',
        action_space=meta['action_space'], agent_type=agent_type,
        init_dice=d.get('init_dice_mean'), final_dice=d.get('final_dice_mean'),
        best_dice_ceiling=d.get('best_dice_mean'), value_floored_dice=d.get('value_floored_dice_mean'),
        delta_dice=d.get('delta_dice_mean'), value_floored_delta=d.get('value_floored_delta_mean'),
        final_hd95=d.get('final_hd95_mean'), init_hd95=d.get('init_hd95_mean'),
        final_iou=d.get('final_iou_mean'), init_iou=d.get('init_iou_mean'), delta_iou=d.get('delta_iou_mean'),
        final_biou=d.get('final_biou_mean'), init_biou=d.get('init_biou_mean'), delta_biou=d.get('delta_biou_mean'),
        final_precision=d.get('final_precision_mean'), final_sensitivity=d.get('final_sensitivity_mean'),
        final_msd=d.get('final_msd_mean'), init_msd=d.get('init_msd_mean'),
        n_refinable=d.get('n_refinable'), n_total=d.get('n_total'), refinable_frac=d.get('refinable_frac'),
        value_floored_dice_refinable=d.get('value_floored_dice_refinable_mean'),
        value_floored_delta_refinable=d.get('value_floored_delta_refinable_mean'),
        routed_dice=d.get('routed_dice_mean'), routed_delta=d.get('routed_delta_mean'),
        reeval=d.get('reeval', False), source_checkpoint=d.get('source_checkpoint'),
    )
    meta = dict(dataset=row['dataset'], class_name=row['class_name'], agent_type=agent_type,
                model=meta['display'], kind='drl')
    return [row], meta

def parse_classifier_summary(d, path):
    return dict(source=str(path), best_val_acc=d.get('best_val_acc'),
                accuracy=d.get('test_metrics', {}).get('accuracy'),
                n_test=d.get('test_metrics', {}).get('n_test'),
                per_class=d.get('test_metrics', {}).get('per_class', {}),
                epochs_trained=d.get('epochs_trained'), class_names=d.get('class_names'))

unet_rows, drl_rows, classifier_summaries, unrecognized_json = [], [], [], []
json_meta = {}   # str(path) -> dict(dataset=, model=, class_name=, agent_type=, kind=)
for path in find_files({'.json'}):
    d = _load_json(path)
    if d is None:
        continue
    if isinstance(d, dict) and 'test_dice' in d:
        rows, meta = parse_unet_summary(d, path)
        unet_rows += rows
        json_meta[str(path)] = meta
    elif isinstance(d, dict) and {'init_dice_mean', 'final_dice_mean'} <= d.keys():
        rows, meta = parse_drl_summary(d, path)
        drl_rows += rows
        json_meta[str(path)] = meta
    elif isinstance(d, dict) and {'accuracy', 'per_class'} <= d.get('test_metrics', {}).keys():
        classifier_summaries.append(parse_classifier_summary(d, path))
    else:
        unrecognized_json.append(path)

unet_df = pd.DataFrame(unet_rows)
drl_df = pd.DataFrame(drl_rows)

print(f'[ingestion] U-Net summary rows: {len(unet_df)}  (files: {unet_df["source"].nunique() if len(unet_df) else 0})')
print(f'[ingestion] DRL summary rows:   {len(drl_df)}  (files: {drl_df["source"].nunique() if len(drl_df) else 0})')
print(f'[ingestion] classifier summaries: {len(classifier_summaries)}')
if unrecognized_json:
    print(f'[ingestion] {len(unrecognized_json)} JSON file(s) matched no known shape, skipped:')
    for p in unrecognized_json:
        print(f'    - {p}')
if unet_df.empty and drl_df.empty and not classifier_summaries:
    print(f"""
No result files found under any of: {[str(r) for r in SEARCH_ROOTS]}.
Drop your *_summary.json / *_reeval_test_results.json / brisc_tumor_classifier_summary.json
files anywhere under a searched root (subfolders OK, or attach as Kaggle datasets), then re-run.
""")


In [ ]:
# ── CSVs: U-Net per-patient test_scores (real paired data!) + generic DRL per-sample ──
unet_csv_frames = []       # list of dict(path, dataset, model, df)
generic_paired_frames = {} # path -> df, for the init_dice/final_dice shape (DRL, if ever exported)
other_csv = []

for path in find_files({'.csv'}):
    try:
        df = pd.read_csv(path)
    except Exception as e:
        print(f'  [skip] {path}: {e}')
        continue
    if 'patient' in df.columns and any(c.startswith('dice_') for c in df.columns):
        meta = sibling_meta(path, ['_test_scores.csv'], json_meta) or {}
        unet_csv_frames.append(dict(path=str(path), dataset=meta.get('dataset', '?'),
                                     model=meta.get('model', '?'), df=df))
    elif {'init_dice', 'final_dice'} <= set(df.columns):
        generic_paired_frames[str(path)] = df
    else:
        other_csv.append(path)

print(f'[ingestion] U-Net per-patient CSVs: {len(unet_csv_frames)}')
for f in unet_csv_frames:
    tag = f'{f["dataset"]}/{f["model"]}' if f['model'] != '?' else '(no sibling summary.json found to tag model)'
    print(f'    - {f["path"]}  [{tag}]  n={len(f["df"])}')
print(f'[ingestion] generic paired (init_dice/final_dice) CSVs: {len(generic_paired_frames)}')
for k in generic_paired_frames:
    print(f'    - {k}')
if other_csv:
    print(f'[ingestion] {len(other_csv)} CSV(s) matched no known shape, skipped:')
    for p in other_csv:
        print(f'    - {p}')


In [ ]:
# ── PNGs: classify by filename, tag via sibling JSON where applicable ──
png_gallery = {'comparison': [], 'behaviour': [], 'learning_curves': [],
               'qualitative': [], 'classifier_confusion': [], 'classifier_curves': [], 'other': []}

for path in find_files({'.png'}):
    name = path.name.lower()
    if 'classifier' in name and 'confusion' in name:
        png_gallery['classifier_confusion'].append(dict(path=path, caption=path.stem))
    elif 'classifier' in name:
        png_gallery['classifier_curves'].append(dict(path=path, caption=path.stem))
    elif name.endswith('_reeval_comparison.png'):
        meta = sibling_meta(path, ['_reeval_comparison.png'], json_meta) or {}
        cap = f"{meta.get('dataset','?')}/{meta.get('class_name','?')} — {meta.get('model','?')} (comparison)"
        png_gallery['comparison'].append(dict(path=path, caption=cap))
    elif name.endswith('_reeval_behaviour.png'):
        meta = sibling_meta(path, ['_reeval_behaviour.png'], json_meta) or {}
        cap = f"{meta.get('dataset','?')}/{meta.get('class_name','?')} — {meta.get('model','?')} (behaviour)"
        png_gallery['behaviour'].append(dict(path=path, caption=cap))
    elif name.endswith('_learning_curves.png'):
        meta = sibling_meta(path, ['_learning_curves.png'], json_meta) or {}
        cap = f"{meta.get('dataset','?')} — {meta.get('model','?')} (learning curves)" if meta else path.stem
        png_gallery['learning_curves'].append(dict(path=path, caption=cap))
    elif name.endswith('_qualitative.png'):
        meta = sibling_meta(path, ['_qualitative.png'], json_meta) or {}
        cap = f"{meta.get('dataset','?')} — {meta.get('model','?')} (qualitative)" if meta else path.stem
        png_gallery['qualitative'].append(dict(path=path, caption=cap))
    else:
        png_gallery['other'].append(dict(path=path, caption=path.stem))

for k, v in png_gallery.items():
    print(f'[ingestion] {k}: {len(v)} image(s)')


In [ ]:
# ── Checkpoints: optional, torch-only, scalar metadata (never rebuilds a model) ──
ckpt_rows = []
if HAVE_TORCH:
    for path in find_files({'.pt', '.pth'}):
        try:
            ck = torch.load(path, map_location='cpu', weights_only=False)
        except Exception as e:
            print(f'  [skip] {path}: could not load ({e})')
            continue
        if not isinstance(ck, dict):
            continue
        ckpt_rows.append(dict(
            path=str(path), filename=path.name,
            step=ck.get('step'), epoch=ck.get('epoch'),
            best_dice=ck.get('best_dice'), val_acc=ck.get('val_acc'),
            class_names=ck.get('class_names'),
        ))
    print(f'[ingestion] checkpoints with readable metadata: {len(ckpt_rows)}')
else:
    print('[ingestion] torch not importable -- skipping checkpoint metadata (Section 9 will note this).')
ckpt_df = pd.DataFrame(ckpt_rows)


## 2. Master tidy comparison table\n\nOne row per (dataset, class, model), Dice as the common column, tagged by `model_family` (UNet / DRL) and `action_space` (n/a / discrete / continuous).

In [ ]:
master_rows = []
if not unet_df.empty:
    for _, r in unet_df.iterrows():
        master_rows.append(dict(dataset=r['dataset'], class_name=r['class_name'],
                                 model=r['model'], model_family='UNet', action_space='n/a',
                                 dice=r['dice'], hd95=r['hd95'], iou=r['iou']))
if not drl_df.empty:
    for _, r in drl_df.iterrows():
        deploy_dice = r['value_floored_dice'] if pd.notna(r.get('value_floored_dice')) else r['final_dice']
        master_rows.append(dict(dataset=r['dataset'], class_name=r['class_name'],
                                 model=r['model'], model_family='DRL', action_space=r['action_space'],
                                 dice=deploy_dice, hd95=r['final_hd95'], iou=r['final_iou']))

master_df = pd.DataFrame(master_rows)
if not master_df.empty:
    master_df = master_df.sort_values(['dataset', 'class_name', 'model_family', 'model']).reset_index(drop=True)
master_df


## 3. U-Net baseline landscape

`LiteUNet` is the deliberately weak baseline the DRL agents refine (chosen because it has
headroom); `AttentionResUNet` is the strong upper-bound competitor. The gap between them,
per class, is the ceiling a DRL agent could theoretically reach.

In [ ]:
if unet_df.empty:
    print('No U-Net summaries loaded -- skipping.')
else:
    pivot = unet_df.pivot_table(index=['dataset', 'class_name'], columns='model', values='dice')
    if {'LiteUNet', 'AttentionResUNet'} <= set(pivot.columns):
        pivot['headroom_to_attn'] = pivot['AttentionResUNet'] - pivot['LiteUNet']
    display(pivot)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    plot_df = unet_df.copy()
    plot_df['label'] = plot_df['dataset'] + ' / ' + plot_df['class_name']
    models = sorted(plot_df['model'].unique())
    labels = sorted(plot_df['label'].unique())
    width = 0.8 / max(len(models), 1)
    for i, m in enumerate(models):
        sub = plot_df[plot_df['model'] == m].set_index('label')['dice'].reindex(labels)
        ax.bar(np.arange(len(labels)) + i * width, sub.values, width, label=m)
    ax.set_xticks(np.arange(len(labels)) + width * (len(models) - 1) / 2)
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_ylabel('Test Dice'); ax.set_title('U-Net baselines by dataset / class'); ax.legend()
    fig.tight_layout(); fig.savefig(FIG_DIR / 'unet_baseline_landscape.png'); plt.show()

    if unet_csv_frames:
        fig, ax = plt.subplots(figsize=(9, 4.5))
        box_data, box_labels = [], []
        for f in unet_csv_frames:
            dice_cols = [c for c in f['df'].columns if c.startswith('dice_')]
            for c in dice_cols:
                vals = f['df'][c].dropna().values
                if len(vals):
                    box_data.append(vals)
                    box_labels.append(f"{f['dataset']}/{c.replace('dice_', '')}\n{f['model']}")
        if box_data:
            ax.boxplot(box_data, labels=box_labels, showmeans=True)
            ax.set_ylabel('Per-patient Dice'); ax.set_title('U-Net per-patient Dice spread')
            plt.xticks(rotation=45, ha='right')
            fig.tight_layout(); fig.savefig(FIG_DIR / 'unet_per_patient_spread.png'); plt.show()


## 4. Discrete vs continuous action space

`DuelingDDQN` acts over discrete contour-sector edits (learned STOP action); `TD3` acts
over continuous per-sector displacements (no learned STOP -- converge-and-hold + a
`fail_thresh` safety net, see `iteris/drl_reeval/re_eval_td3.py`). Drift (`best_dice_ceiling`
vs `final_dice` gap) is reported explicitly since only TD3 is structurally exposed to it.

In [ ]:
if drl_df.empty:
    print('No DRL summaries loaded -- skipping.')
else:
    drl_df['drift'] = drl_df['best_dice_ceiling'] - drl_df['final_dice']
    drl_df['deploy_delta'] = drl_df['value_floored_delta'].fillna(drl_df['delta_dice'])

    agg = drl_df.groupby('action_space').agg(
        n_runs=('model', 'size'), mean_deploy_delta=('deploy_delta', 'mean'),
        std_deploy_delta=('deploy_delta', 'std'),
        win_rate=('deploy_delta', lambda s: float((s > 0).mean())),
        mean_drift=('drift', 'mean'), mean_final_hd95=('final_hd95', 'mean'),
    ).round(4)
    display(agg)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    drl_df.boxplot(column='deploy_delta', by='action_space', ax=axes[0])
    axes[0].axhline(0, color='r', ls='--', lw=1)
    axes[0].set_title('Deployable delta-Dice by action space'); axes[0].set_ylabel('delta-Dice (value-floored)')
    drl_df.boxplot(column='drift', by='action_space', ax=axes[1])
    axes[1].set_title('Drift: best-seen minus final (deploy) Dice'); axes[1].set_ylabel('delta-Dice')
    fig.suptitle(''); plt.tight_layout()
    fig.savefig(FIG_DIR / 'discrete_vs_continuous.png'); plt.show()


## 5. Per-class / per-dataset learning behaviour

Positive deployable delta = genuine learning; near-zero = the STOP-at-baseline collapse
described in the project's PBRS fix notes; negative = actively hurting the baseline.

In [ ]:
if drl_df.empty:
    print('No DRL summaries loaded -- skipping.')
else:
    heat = drl_df.pivot_table(index=['dataset', 'class_name'], columns='model', values='deploy_delta')
    try:
        display(heat.style.background_gradient(cmap='RdYlGn', axis=None, vmin=-0.05, vmax=0.05))
    except Exception:
        display(heat)

    fig, ax = plt.subplots(figsize=(1.6 * max(len(heat.columns), 2) + 2, 0.5 * len(heat) + 2))
    im = ax.imshow(heat.values, cmap='RdYlGn', vmin=-0.05, vmax=0.05, aspect='auto')
    ax.set_xticks(range(len(heat.columns))); ax.set_xticklabels(heat.columns, rotation=30, ha='right')
    ax.set_yticks(range(len(heat.index))); ax.set_yticklabels([f'{d}/{c}' for d, c in heat.index])
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            v = heat.values[i, j]
            if pd.notna(v):
                ax.text(j, i, f'{v:+.3f}', ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax, label='Deployable delta-Dice vs U-Net baseline')
    ax.set_title('Learning behaviour: deployable delta-Dice by dataset/class x agent')
    fig.tight_layout(); fig.savefig(FIG_DIR / 'learning_behaviour_heatmap.png'); plt.show()

    print()
    print('Best-learning (dataset, class) per agent (highest deployable delta-Dice):')
    for m in drl_df['model'].unique():
        sub = drl_df[drl_df['model'] == m].sort_values('deploy_delta', ascending=False)
        if not sub.empty:
            top = sub.iloc[0]
            print(f'  {m:>14s}: {top["dataset"]}/{top["class_name"]}  delta-Dice={top["deploy_delta"]:+.4f}')


## 6. Most-improved DRL agent ranking

Ranks every (agent, dataset, class) run by deployable delta-Dice. See Section 3's U-Net
headroom and Section 11's caveats before over-interpreting a small or negative result.

In [ ]:
if drl_df.empty:
    print('No DRL summaries loaded -- skipping.')
else:
    ranked = drl_df.sort_values('deploy_delta', ascending=False)[
        ['dataset', 'class_name', 'model', 'action_space', 'init_dice', 'final_dice',
         'value_floored_dice', 'deploy_delta', 'best_dice_ceiling', 'drift', 'refinable_frac']
    ].reset_index(drop=True)
    display(ranked)

    per_agent = drl_df.groupby('model').agg(
        n_runs=('deploy_delta', 'size'), mean_deploy_delta=('deploy_delta', 'mean'),
        median_deploy_delta=('deploy_delta', 'median'),
        n_positive=('deploy_delta', lambda s: int((s > 0).sum())),
    ).sort_values('mean_deploy_delta', ascending=False).round(4)
    display(per_agent)

    if not per_agent.empty:
        winner = per_agent.index[0]
        print()
        print(f'>>> Most successful DRL agent by mean deployable delta-Dice across all loaded runs: '
              f'{winner}  (mean delta-Dice={per_agent.loc[winner, "mean_deploy_delta"]:+.4f}, '
              f'{per_agent.loc[winner, "n_positive"]}/{per_agent.loc[winner, "n_runs"]} runs beat baseline)')


## 7. Qualitative visual gallery

Every comparison/behaviour/learning-curve/qualitative-grid image found, displayed inline.
This is the "does it actually look right" check that aggregate Dice numbers can hide —
e.g. a positive delta-Dice from a mask that's technically closer to GT but visibly
mangled (holes, disconnected blobs) would show up here, not in Section 6's table.

In [ ]:
def _show_gallery(entries, ncols=2, figsize_per=(6, 5)):
    if not entries:
        return
    n = len(entries)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_per[0]*ncols, figsize_per[1]*nrows))
    axes = np.atleast_1d(axes).flatten()
    for ax, entry in zip(axes, entries):
        try:
            img = mpimg.imread(entry['path'])
            ax.imshow(img)
        except Exception as e:
            ax.text(0.5, 0.5, f'could not load:\n{e}', ha='center', va='center', wrap=True)
        ax.set_title(entry['caption'], fontsize=9)
        ax.axis('off')
    for ax in axes[len(entries):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()

for key, title in [('comparison', 'DRL init vs refined vs GT (comparison plots)'),
                    ('behaviour', 'DRL per-episode Dice trajectories (behaviour plots)'),
                    ('learning_curves', 'U-Net training curves'),
                    ('qualitative', 'U-Net qualitative prediction grids')]:
    entries = png_gallery.get(key, [])
    print(f'--- {title}: {len(entries)} image(s) ---')
    _show_gallery(entries)

if png_gallery.get('other'):
    print(f"{len(png_gallery['other'])} other PNG(s) found (not matching a known naming pattern):")
    for e in png_gallery['other']:
        print(f'    - {e["path"]}')


## 8. BRISC tumor-type classifier evaluation

The classifier (`iteris.classifier.TumorClassifier`) predicts tumor type
(glioma / meningioma / pituitary / non_tumor) — a separate task from the
segmentation models above, evaluated on accuracy + per-class precision/recall/F1
rather than Dice.

In [ ]:
if not classifier_summaries:
    print("""
No brisc_tumor_classifier_summary.json found -- skipping. Run the eval cell in the
classifier notebook (loads the trained checkpoint, calls
iteris.classifier.evaluate_classifier_test_set, then save_classifier_summary_json)
and drop its output under a searched root.
""")
else:
    for summ in classifier_summaries:
        print(f"Source: {summ['source']}")
        print(f"  Best val accuracy: {summ['best_val_acc']}   "
              f"Test accuracy: {summ['accuracy']}  (n={summ['n_test']})   "
              f"Epochs trained: {summ['epochs_trained']}")
        pc = pd.DataFrame(summ['per_class']).T
        display(pc)

        fig, ax = plt.subplots(figsize=(7, 4))
        pc[['precision', 'recall', 'f1']].plot(kind='bar', ax=ax)
        ax.set_ylim(0, 1.05); ax.set_ylabel('Score'); ax.set_title('Per-class precision / recall / F1')
        plt.xticks(rotation=20, ha='right')
        fig.tight_layout(); fig.savefig(FIG_DIR / 'classifier_per_class_scores.png'); plt.show()

    print(f"--- Confusion matrix image(s): {len(png_gallery['classifier_confusion'])} ---")
    _show_gallery(png_gallery['classifier_confusion'], ncols=1, figsize_per=(6, 6))
    print(f"--- Learning curve image(s): {len(png_gallery['classifier_curves'])} ---")
    _show_gallery(png_gallery['classifier_curves'], ncols=1, figsize_per=(9, 4))


## 9. Checkpoint / training provenance

Scalar metadata only (step / epoch / best_dice / val_acc), read straight out of the
`.pt` files with plain `torch.load` — never rebuilds a model, so this works without
`iteris` or `monai` installed. Useful for sanity-checking which checkpoint got
deployed (`*_best.pt`) vs how far training actually ran (`*_stepN.pt` milestones).

In [ ]:
if not HAVE_TORCH:
    print('torch is not importable in this environment -- skipping (this section is optional).')
elif ckpt_df.empty:
    print('No .pt/.pth checkpoints found under the searched roots.')
else:
    display(ckpt_df.sort_values('path'))


## 10. Statistical significance (Wilcoxon signed-rank + Bonferroni)

Two sources of real paired data are used, both tested here with the same
Bonferroni correction applied jointly across every test run in this section:

1. **U-Net per-patient CSVs** (`*_test_scores.csv`) — if two U-Net models were
   evaluated on the same dataset (e.g. LiteUNet vs AttentionResUNet), their
   per-patient Dice is paired by `patient` ID and tested per class. This data
   already exists automatically from `evaluate_test_set` — nothing extra to export.
2. **Generic paired CSVs** (`init_dice`/`final_dice` columns) — for DRL runs, if
   you exported per-sample data (see Section intro / `results/README.md`).

Aggregate JSON summaries alone (Sections 1-6) are already means and can't support
a real hypothesis test on their own.

In [ ]:
sig_rows = []

# (1) U-Net per-patient, paired across model pairs sharing a dataset.
by_dataset = {}
for f in unet_csv_frames:
    by_dataset.setdefault(f['dataset'], []).append(f)
for dataset, frames in by_dataset.items():
    for i in range(len(frames)):
        for j in range(i + 1, len(frames)):
            a, b = frames[i], frames[j]
            if a['model'] == b['model']:
                continue
            merged = a['df'].merge(b['df'], on='patient', suffixes=('_a', '_b'))
            dice_cols = [c for c in a['df'].columns if c.startswith('dice_')]
            for c in dice_cols:
                ca, cb = f'{c}_a', f'{c}_b'
                if ca not in merged.columns or cb not in merged.columns:
                    continue
                paired = merged[[ca, cb]].dropna()
                if len(paired) < 5:
                    continue
                stat, p = stats.wilcoxon(paired[cb], paired[ca])
                sig_rows.append(dict(
                    comparison=f"{dataset}/{c.replace('dice_', '')}: {b['model']} vs {a['model']}",
                    n=len(paired), mean_delta=(paired[cb] - paired[ca]).mean(),
                    wilcoxon_stat=stat, p_raw=p))

# (2) Generic paired CSVs (DRL per-sample, if ever exported).
for path, df in generic_paired_frames.items():
    paired = df[['init_dice', 'final_dice']].dropna()
    if len(paired) < 5:
        print(f'  [skip] {path}: only {len(paired)} paired samples, too few for Wilcoxon')
        continue
    stat, p = stats.wilcoxon(paired['final_dice'], paired['init_dice'])
    sig_rows.append(dict(comparison=path, n=len(paired),
                          mean_delta=(paired['final_dice'] - paired['init_dice']).mean(),
                          wilcoxon_stat=stat, p_raw=p))

if sig_rows:
    sig_df = pd.DataFrame(sig_rows)
    sig_df['p_bonferroni'] = np.minimum(sig_df['p_raw'] * len(sig_df), 1.0)
    sig_df['significant_at_0.05'] = sig_df['p_bonferroni'] < 0.05
    display(sig_df)
else:
    print("""No paired data available for significance testing yet.
Either: (a) no two U-Net models share a dataset with matching per-patient CSVs, or
(b) no *_test_scores.csv / generic paired CSVs were found at all. Drop the
*_test_scores.csv files evaluate_test_set already produces for each U-Net run,
and/or per-sample DRL exports (see the notebook intro), then re-run.""")


## 11. Export & auto-generated summary

Saves the master table and prints a summary built entirely from computed values —
nothing here is pre-written prose.

**Context for interpreting a "DRL didn't beat baseline" result** (from this project's
own diagnostics, `docs/CONTEXT.md`): three separate, non-overlapping caps can each
produce that outcome, and only the last is an optimization problem the training
recipe can fix -- (a) **representation cap**: the deployed mask is a smoothed
periodic-spline approximation of the largest connected component, never the raw
U-Net mask, so even the do-nothing policy pays a tax vs. the U-Net benchmark;
(b) **information cap**: a GT-blind refiner has no privileged signal the U-Net
lacked; (c) **optimization cap**: reward shaping / STOP-timing / training budget.
Run `iteris.diagnostics.headroom_report` to tell these apart for any run showing
near-zero or negative delta-Dice.

In [ ]:
if not master_df.empty:
    out_csv = OUT_DIR / 'master_comparison.csv'
    master_df.to_csv(out_csv, index=False)
    print(f'[export] master comparison table -> {out_csv}')

print()
print('=' * 70)
print('SUMMARY (auto-generated from loaded results)')
print('=' * 70)

if unet_df.empty and drl_df.empty and not classifier_summaries:
    print('No result files loaded yet -- nothing to summarize. See Section 1.')
else:
    if not unet_df.empty:
        best_unet = unet_df.sort_values('dice', ascending=False).iloc[0]
        print(f'- Strongest U-Net baseline observed: {best_unet["model"]} on '
              f'{best_unet["dataset"]}/{best_unet["class_name"]} (Dice={best_unet["dice"]:.4f}).')
    if not drl_df.empty:
        agent_summary = drl_df.groupby('action_space')['deploy_delta'].mean()
        for space, val in agent_summary.items():
            print(f'- Mean deployable delta-Dice, {space} action space: {val:+.4f}')
        if len(agent_summary) >= 2:
            better_space = agent_summary.idxmax()
            print(f'  -> {better_space} action space shows the larger average deployable gain.')
        per_agent = drl_df.groupby('model')['deploy_delta'].mean().sort_values(ascending=False)
        print(f'- Best-performing DRL agent overall: {per_agent.index[0]} '
              f'(mean delta-Dice={per_agent.iloc[0]:+.4f}).')
        best_cell = drl_df.sort_values('deploy_delta', ascending=False).iloc[0]
        print(f'- Single best (agent, dataset, class) result: {best_cell["model"]} on '
              f'{best_cell["dataset"]}/{best_cell["class_name"]} (delta-Dice={best_cell["deploy_delta"]:+.4f}).')
        n_beat = int((drl_df['deploy_delta'] > 0).sum())
        print(f'- {n_beat}/{len(drl_df)} loaded DRL runs beat their U-Net baseline (deployable comparison).')
    if classifier_summaries:
        accs = [s['accuracy'] for s in classifier_summaries if s['accuracy'] is not None]
        if accs:
            print(f'- BRISC tumor-type classifier test accuracy: {accs[0]:.4f}'
                  + (f' (mean over {len(accs)} runs)' if len(accs) > 1 else '') + '.')
    n_gallery = sum(len(v) for v in png_gallery.values())
    print(f'- Qualitative gallery: {n_gallery} image(s) displayed in Section 7/8.')
    if HAVE_TORCH and not ckpt_df.empty:
        print(f'- Checkpoint provenance: {len(ckpt_df)} .pt file(s) inspected in Section 9.')
print('=' * 70)
